# 06c. MLP: преобразование признаков токена

Лекция: [MLP в Transformer](../../notes/interview-prep/08i-token-mlp.md).

Начни с разделов 1–5 лекции и части A. В этом уроке только MLP; готовый attention используем при сборке блока в следующем уроке.

Формат работы: ментор читает сохранённый код и ответы, не запускает ноутбук или проверки и не делает коммит без отдельной просьбы.

In [1]:
import torch
from torch import nn

torch.manual_seed(42)
d_model = 4
d_ff = 16
x = torch.randn(2, 3, d_model)  # Один batch: 2 последовательности по 3 токена.


## A. Проследить путь одного batch

Заполни два Linear: первый расширяет 4 признака до 16, второй возвращает 16 к 4. bias оставь включённым (значение по умолчанию).

GELU уже добавлен. Он изменяет значения, но не форму. Пока не нужно писать свой класс.

In [2]:
expand = nn.Linear(d_model, d_ff)
activation = nn.GELU()
contract = nn.Linear(d_ff, d_model)

z = expand(x)
h = activation(z)
y = contract(h)

print('Вход:', x.shape)
print('После первого Linear:', z.shape)
print('После GELU:', h.shape)
print('После второго Linear:', y.shape)
print('Первый weight:', expand.weight.shape)
print('Второй weight:', contract.weight.shape)
print('Один токен до GELU:', z[0, 0])
print('Тот же токен после GELU:', h[0, 0])


Вход: torch.Size([2, 3, 4])
После первого Linear: torch.Size([2, 3, 16])
После GELU: torch.Size([2, 3, 16])
После второго Linear: torch.Size([2, 3, 4])
Первый weight: torch.Size([16, 4])
Второй weight: torch.Size([4, 16])
Один токен до GELU: tensor([-0.6815, -0.0381, -0.4272, -0.7105,  0.9546, -0.2639,  1.6376, -0.8695,
         0.8444, -1.5267, -0.6684,  1.3174,  0.1823,  0.8753,  1.2748,  0.0394],
       grad_fn=<SelectBackward0>)
Тот же токен после GELU: tensor([-0.1689, -0.0185, -0.1430, -0.1696,  0.7924, -0.1045,  1.5545, -0.1672,
         0.6762, -0.0968, -0.1684,  1.1938,  0.1043,  0.7084,  1.1458,  0.0203],
       grad_fn=<SelectBackward0>)


### Остановка A

1. После первого Linear стало больше токенов или признаков у каждого токена?
2. Зачем возвращаем число признаков к d_model?
3. Почему между двумя Linear нужна нелинейность?

**Мои ответы:**

1. Токенов не стало больше, для кадого токена стало 16 признаков (внутри ffn)
2. Чтобы можно было суммировать со входом в residual connections
3. Потому что если ее убрать, то любое количество линейных преобразований можно свести к одному

Сохрани и пришли на разбор. Часть B сделаем после A.

**Разбор ментора (исходные ответы сохранены):**

Все три ответа верны. Число токенов сохраняется, ширина признаков меняется 4 → 16 → 4. Возврат к d_model нужен для residual-сложения. С bias преобразования называются аффинными; их композиция без нелинейности тоже сводится к одному Linear с bias. Код и сохранённые формы соответствуют заданию.


## B. Один и тот же MLP для всех позиций

Готовый опыт. Sequential использует те самые слои из A, не создаёт их копии. Изменим последний токен первой последовательности и посмотрим, затронут ли первый.

Здесь только MLP, без attention. Поэтому разные позиции обрабатываются независимо.

In [3]:
mlp = nn.Sequential(expand, activation, contract)
original_output = mlp(x)

changed_x = x.clone()
changed_x[0, -1] = changed_x[0, -1] + 5
changed_output = mlp(changed_x)

print('Первый токен до изменения соседа:', original_output[0, 0])
print('Первый токен после изменения соседа:', changed_output[0, 0])
print('Последний токен до:', original_output[0, -1])
print('Последний токен после:', changed_output[0, -1])
print('Число параметров MLP:', sum(p.numel() for p in mlp.parameters()))

# Соединяем с известными компонентами; это только MLP-ветвь, не весь блок.
norm = nn.LayerNorm(d_model)
residual_output = x + mlp(norm(x))
print('После residual:', residual_output.shape)


Первый токен до изменения соседа: tensor([ 0.1655,  0.7260,  0.0831, -0.4216], grad_fn=<SelectBackward0>)
Первый токен после изменения соседа: tensor([ 0.1655,  0.7260,  0.0831, -0.4216], grad_fn=<SelectBackward0>)
Последний токен до: tensor([ 0.2903,  0.1467,  0.2353, -0.2189], grad_fn=<SelectBackward0>)
Последний токен после: tensor([-0.3990, -0.8336, -0.4183, -0.1920], grad_fn=<SelectBackward0>)
Число параметров MLP: 148
После residual: torch.Size([2, 3, 4])


### Остановка B

1. Почему первый выходной токен не изменился, хотя последний входной изменился?
2. Посчитай параметры двух Linear вручную, включая bias. GELU добавляет параметры?
3. Понадобятся ли новые параметры MLP, если вместо T=3 взять T=20 при том же d_model?

**Мои ответы:**

1. Потому что у нас нет attention, токены независимы. А последний мы сами изменили changed_x[0, -1]
2. GELU не добавляет.  4\*16\*2+4+16
3. Нет, не понадобятся

Здесь нет backward и optimizer.step, обучения не было. После разбора соберём полный pre-norm Transformer-блок.

**Разбор ментора (исходные ответы сохранены):**

Все три ответа верны. Точнее, выход MLP в данной позиции зависит только от входного вектора этой позиции и общих параметров MLP. В полном Transformer вход уже может содержать контекст после attention. Подсчёт: первый Linear — 16*4+16=80, второй — 4*16+4=68; всего 148, как в сохранённом выводе. У GELU параметров нет. При T=20 применяем те же веса к большему числу токенов; новых параметров не нужно. LayerNorm в конце — отдельный модуль с 8 параметрами, они не входят в 148 параметров MLP. Просмотрены код, ответы и сохранённые outputs без запуска ноутбука или проверок.
